# Q2 — Finetune VGG19 on the 15-class dataset

This notebook finetunes a pretrained VGG19. It includes utilities to partially unfreeze layers and compute per-class precision & recall.

In [ ]:
# Setup for Colab and local environments
import os, sys
from pathlib import Path

# Detect if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Clone repo to get dataset
    os.system('git clone https://github.com/nelsunnel/LLMs-and-GenAI-Assignment.git /content/project')
    os.chdir('/content/project')
    # Install requirements
    os.system('pip install -q -r requirements.txt')
    PROJECT_ROOT = Path('/content/project')
else:
    print('Running locally')
    PROJECT_ROOT = Path('.')

print('Project root:', PROJECT_ROOT)

In [ ]:
from pathlib import Path
import re
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import precision_recall_fscore_support

DATA_ROOT = PROJECT_ROOT / 'Datasets' / 'dataset'
if not DATA_ROOT.exists():
    raise RuntimeError('Dataset not found — adjust DATA_ROOT')
CLASSES = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])
NUM_CLASSES = len(CLASSES)

def build_splits(root, classes):
    import re
    train, test = [], []
    num_re = re.compile(r'(\d+)')
    for idx, c in enumerate(classes):
        p = Path(root)/c
        imgs = sorted([x for x in p.iterdir() if x.suffix.lower() in ['.jpg','.png','.jpeg']])
        for im in imgs:
            m = num_re.search(im.stem)
            if m and 1 <= int(m.group(1)) <= 40:
                train.append((str(im), idx))
            else:
                test.append((str(im), idx))
    return train, test

train_items, test_items = build_splits(DATA_ROOT, CLASSES)
print(len(train_items), len(test_items))

IMG_SIZE = 224
train_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
test_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class SimpleImageDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

train_ds = SimpleImageDataset(train_items, transform=train_tf)
test_ds = SimpleImageDataset(test_items, transform=test_tf)
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device', device)